# Algerian Forest Fires — Solution Notebook (Extended)

**Role:** Data Scientist as *Inference Specialist*

Complete worked solution with alternate implementations, residual diagnostics, extra visualisations, simulation helpers, optimisation notes and domain-transfer examples.

Dataset: 243 observations from Bejaia & Sidi Bel-abbes, Algeria (UCI / AI2SD 2019).


## Flowchart: Desired Outcome for the Algerian Forest Fires Analysis

This flowchart describes the complete Inference-Specialist workflow for turning the forests weather/fire dataset into actionable insights about humidity, fuel moisture (FFMC) and fire-weather potential (FWI).

```mermaid
flowchart TD
    A[Start: Inference Questions<br/>• How does temp affect humidity by region?<br/>• Does the temp→FFMC slope differ by fire occurrence?<br/>• Is the FFMC–humidity relationship linear or curved?<br/>• Can weather vars jointly predict FFMC / FWI?] --> B[Load & Inspect Data<br/>forests.csv – 243 rows, 2 regions]
    B --> C[Assumption Check<br/>Correlation heatmap / VIF<br/>Flag collinear predictors]
    C --> D1[EDA Scatters<br/>humid ~ temp | region<br/>FFMC ~ temp | fire<br/>FFMC ~ humid]
    D1 --> E1[MLR: humid ~ temp + region<br/>Parallel lines model]
    D1 --> E2[MLR + Interaction<br/>FFMC ~ temp * fire<br/>Different slopes]
    D1 --> E3[Polynomial<br/>FFMC ~ humid + humid²]
    E1 & E2 & E3 --> F[Interpret Coefficients<br/>Context + practical meaning<br/>Write region-/fire-specific equations]
    F --> G[Visualise Fitted Lines<br/>Overlay on scatter plots]
    G --> H[Extended Models<br/>FFMC ~ temp+rain+wind+humid<br/>FWI ~ ISI + BUI]
    H --> I[Diagnostics & Insights<br/>Residuals, R², collinearity of derived indices]
    I --> J[Simulation / Sensitivity<br/>What-if temp/humidity changes<br/>Bootstrap or Monte-Carlo predictions]
    J --> K[Transferability Note<br/>Same pipeline → any domain<br/>just swap the CSV]
    K --> L[Report & Recommendations<br/>1-page summary for stakeholders]
```

**Inference Specialist mindset:** We care less about pure predictive accuracy and more about *interpretable coefficients*, assumption validity, interaction effects, and whether a simple linear story is enough or a polynomial/interaction is required.


## 0. Setup – Import libraries & load data

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

# nicer default style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('colorblind')

forests = pd.read_csv('forests.csv')
print('Shape:', forests.shape)
print('Columns:', forests.columns.tolist())
print('\nRegion counts:\n', forests['region'].value_counts())
print('\nFire counts:\n', forests['fire'].value_counts())
print('\nNumeric summary:')
print(forests.describe().round(2))


## 1. Multicollinearity check (heatmap)

**Insight:** DMC, DC, BUI and FWI are highly correlated (as expected – FWI is a composite of ISI & BUI, and BUI itself is built from DMC & DC). Never put more than one of these derived indices together as simultaneous predictors unless you are deliberately studying the composite relationship.


In [ ]:
corr_grid = forests.select_dtypes(include=[np.number]).corr()
plt.figure(figsize=(9, 7))
sns.heatmap(corr_grid, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            annot_kws={'size': 8}, linewidths=0.5)
plt.title('Correlation Heatmap – Weather & Fire-Weather Indices')
plt.tight_layout()
plt.show()
plt.clf()

# Highlight strongest pairs
print('Strongest absolute correlations (excl. self):')
c = corr_grid.unstack().abs().sort_values(ascending=False)
print(c[c < 1].drop_duplicates().head(10))


## 2. Humidity vs Temperature by Region (EDA)

The negative relationship is visible in both regions; Sidi Bel-abbes points sit systematically a little lower (drier) for the same temperature.


In [ ]:
sns.lmplot(x='temp', y='humid', hue='region', data=forests, fit_reg=False,
           height=5, aspect=1.3, scatter_kws={'alpha': 0.6})
plt.title('Relative Humidity vs Temperature by Region')
plt.show()
plt.clf()


## 3–6. Multiple regression: humid ~ temp + region

### Coefficients (approximate)
- Intercept ≈ 142.58 (Bejaia reference)
- region[T.Sidi Bel-abbes] ≈ –7.25
- temp ≈ –2.39

**Full equation:**  
`humid = 142.58 – 2.39·temp – 7.25·I(Sidi Bel-abbes)`

**Bejaia:** `humid = 142.58 – 2.39·temp`  
**Sidi Bel-abbes:** `humid = 135.33 – 2.39·temp`

**Interpretation of temp:** Holding region fixed, each +1 °C is associated with a 2.39 percentage-point *decrease* in relative humidity.

**Intercepts:** Extrapolated humidity at 0 °C. Values >100 % are not physically meaningful; they simply tell us the Bejaia line starts higher. Always interpret intercepts inside the observed data range when possible.


In [ ]:
modelH = sm.OLS.from_formula('humid ~ temp + region', data=forests).fit()
print(modelH.params.round(4))
print(modelH.summary().tables[1])

# Overlay fitted lines
sns.lmplot(x='temp', y='humid', hue='region', data=forests, fit_reg=False,
           height=5, aspect=1.3, scatter_kws={'alpha': 0.55})
temps = np.linspace(forests.temp.min(), forests.temp.max(), 60)
p = modelH.params
plt.plot(temps, p['Intercept'] + p['temp']*temps, color='C0', lw=3, label='Bejaia fit')
plt.plot(temps, p['Intercept'] + p['region[T.Sidi Bel-abbes]'] + p['temp']*temps,
         color='C1', lw=3, label='Sidi Bel-abbes fit')
plt.legend()
plt.title('Humidity ~ Temperature + Region (parallel slopes)')
plt.show()
plt.clf()


### Alternate code for the same model

```python
# 1. Explicit C() coding
modelH_alt1 = sm.OLS.from_formula('humid ~ temp + C(region)', data=forests).fit()

# 2. Dummy-matrix form
dummies = pd.get_dummies(forests['region'], drop_first=True)
X = pd.concat([forests[['temp']], dummies], axis=1)
X = sm.add_constant(X)
modelH_alt2 = sm.OLS(forests['humid'], X).fit()

# 3. sklearn (coefficients match after encoding)
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import OneHotEncoder
# ... encode then lr.fit(X, y)
```


## 7–11. FFMC – Interaction model with fire

The scatter suggests steeper positive slope for the No-Fire group and a flatter (almost horizontal) relationship once a fire has occurred. An interaction term is therefore warranted.

**Full equation:**  
`FFMC = –8.11 + 2.45·temp + 76.79·I(Fire) – 1.89·temp·I(Fire)`

**No Fire:** `FFMC = –8.11 + 2.45·temp`  (steep)  
**Fire:** `FFMC = 68.68 + 0.56·temp`  (shallow)

Interpretation: when no fire occurred, each extra degree of temperature is associated with a ~2.45-point rise in FFMC. When a fire *did* occur, the same temperature increase is associated with only a ~0.56-point rise – the fuel was already very dry.


In [ ]:
sns.lmplot(x='temp', y='FFMC', hue='fire', data=forests, fit_reg=False,
           height=5, aspect=1.3, scatter_kws={'alpha': 0.55})
plt.title('FFMC vs Temperature by Fire Occurrence')
plt.show()
plt.clf()

modelF = sm.OLS.from_formula('FFMC ~ temp * fire', data=forests).fit()  # * expands to interaction
print(modelF.params.round(4))
print(modelF.summary().tables[1])

# Overlay non-parallel lines
sns.lmplot(x='temp', y='FFMC', hue='fire', data=forests, fit_reg=False,
           height=5, aspect=1.3, scatter_kws={'alpha': 0.5})
temps = np.linspace(22, 42, 60)
pF = modelF.params
plt.plot(temps, pF['Intercept'] + pF['temp']*temps, color='C0', lw=3, label='No Fire')
plt.plot(temps, pF['Intercept'] + pF['fire[T.True]'] + (pF['temp'] + pF['temp:fire[T.True]'])*temps,
         color='C1', lw=3, label='Fire')
plt.legend()
plt.title('FFMC ~ Temperature * Fire (interaction)')
plt.show()
plt.clf()


### Alternate interaction coding

```python
# Explicit interaction term
modelF2 = sm.OLS.from_formula('FFMC ~ temp + fire + temp:fire', data=forests).fit()

# Manual construction
forests['temp_x_fire'] = forests['temp'] * forests['fire'].astype(int)
X = sm.add_constant(forests[['temp', 'fire']].assign(fire=forests.fire.astype(int)))
X['temp_x_fire'] = forests['temp_x_fire']
modelF3 = sm.OLS(forests['FFMC'], X).fit()
```


## 12–15. FFMC – Quadratic polynomial in humidity

A straight line would miss the clear downward curvature at higher humidity. The quadratic term is highly significant.

**Equation:** `FFMC = 77.63 + 0.752·humid – 0.0114·humid²`

Predicted FFMC:
- 25 % → ≈ 89.3
- 35 % → ≈ 90.0  (tiny increase)
- 60 % → ≈ 81.7
- 70 % → ≈ 74.4  (steep drop)

Interpretation: at low humidity the relationship is almost flat; after ~35 % further increases in humidity produce progressively larger *decreases* in FFMC (fuel becomes wetter, lower fire risk).


In [ ]:
sns.lmplot(x='humid', y='FFMC', data=forests, fit_reg=False, height=5, aspect=1.3,
           scatter_kws={'alpha': 0.5, 'color': 'steelblue'})
plt.title('FFMC vs Relative Humidity (raw scatter)')
plt.show()
plt.clf()

modelP = sm.OLS.from_formula('FFMC ~ humid + np.power(humid, 2)', data=forests).fit()
print(modelP.params.round(5))

h_vals = [25, 35, 60, 70]
for h in h_vals:
    pred = modelP.params.iloc[0] + modelP.params.iloc[1]*h + modelP.params.iloc[2]*h**2
    print(f'Humidity {h}% → predicted FFMC = {pred:.2f}')

# Visualise the curve
sns.lmplot(x='humid', y='FFMC', data=forests, fit_reg=False, height=5, aspect=1.3,
           scatter_kws={'alpha': 0.45})
h = np.linspace(20, 90, 120)
plt.plot(h, modelP.params.iloc[0] + modelP.params.iloc[1]*h + modelP.params.iloc[2]*h**2,
         color='darkred', lw=3, label='Quadratic fit')
plt.legend()
plt.title('FFMC ~ Humidity + Humidity²')
plt.show()
plt.clf()


### Alternate polynomial code

```python
# numpy.polyfit (degree 2)
coefs = np.polyfit(forests.humid, forests.FFMC, 2)  # highest power first
print(coefs)  # [a, b, c] for a*x² + b*x + c

# sklearn PolynomialFeatures
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
poly = PolynomialFeatures(degree=2, include_bias=True)
X_poly = poly.fit_transform(forests[['humid']])
lr = LinearRegression().fit(X_poly, forests['FFMC'])
```


## 16. Multi-predictor models

In [ ]:
modelFFMC = sm.OLS.from_formula('FFMC ~ temp + rain + wind + humid', data=forests).fit()
print('=== FFMC ~ temp + rain + wind + humid ===')
print(modelFFMC.params.round(4))
print(f'R² = {modelFFMC.rsquared:.3f}')
print(modelFFMC.summary().tables[1])

modelFWI = sm.OLS.from_formula('FWI ~ ISI + BUI', data=forests).fit()
print('\n=== FWI ~ ISI + BUI ===')
print(modelFWI.params.round(4))
print(f'R² = {modelFWI.rsquared:.3f}  (very high – expected, FWI is derived from these)')


**How to decide on interactions:** Look at residual plots stratified by each continuous predictor, or formally test product terms one at a time and keep those that are both statistically and practically significant. Also examine partial-regression (added-variable) plots.


## Extended A – Residual diagnostics & VIF

In [ ]:
# Residuals for modelH
fitted = modelH.fittedvalues
resid = modelH.resid

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].scatter(fitted, resid, alpha=0.5)
axes[0].axhline(0, color='red', ls='--')
axes[0].set_xlabel('Fitted values')
axes[0].set_ylabel('Residuals')
axes[0].set_title('Residuals vs Fitted (modelH)')

axes[1].hist(resid, bins=20, edgecolor='k', alpha=0.7)
axes[1].set_title('Residual histogram')
plt.tight_layout()
plt.show()
plt.clf()

# VIF example – collinear fire-weather indices
cols = ['DMC', 'DC', 'BUI', 'ISI']
X_vif = sm.add_constant(forests[cols])
vif = pd.DataFrame({
    'variable': X_vif.columns,
    'VIF': [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]
})
print('VIF (values >> 5-10 indicate problematic collinearity):')
print(vif.round(2))


## Extended B – More visualisations

In [ ]:
# Pair-plot of key variables coloured by fire
sns.pairplot(forests[['temp', 'humid', 'FFMC', 'FWI', 'fire']],
             hue='fire', diag_kind='hist', plot_kws={'alpha': 0.5, 's': 20},
             height=2.2)
plt.suptitle('Pair-plot coloured by Fire', y=1.02)
plt.show()
plt.clf()

# Box-plots
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
sns.boxplot(x='region', y='FWI', data=forests, ax=axes[0])
axes[0].set_title('FWI by Region')
sns.boxplot(x='fire', y='FFMC', data=forests, ax=axes[1])
axes[1].set_title('FFMC by Fire Occurrence')
plt.tight_layout()
plt.show()
plt.clf()

# Actual vs Predicted for FWI model
pred_fwi = modelFWI.fittedvalues
plt.figure(figsize=(5.5, 5))
plt.scatter(forests['FWI'], pred_fwi, alpha=0.5)
lims = [0, forests['FWI'].max()]
plt.plot(lims, lims, 'r--', lw=2, label='Perfect fit')
plt.xlabel('Observed FWI')
plt.ylabel('Predicted FWI')
plt.title(f'FWI model – Actual vs Predicted (R²={modelFWI.rsquared:.3f})')
plt.legend()
plt.tight_layout()
plt.show()
plt.clf()


## Simulation / Sensitivity Section

We build simple prediction helpers and a Monte-Carlo envelope around the humidity model. Change the temperature grid or the residual SD to see different risk profiles.


In [ ]:
def predict_humid(temp, region='Bejaia', model=modelH):
    """Point prediction from modelH."""
    p = model.params
    base = p['Intercept'] + p['temp'] * temp
    if region == 'Sidi Bel-abbes':
        base += p['region[T.Sidi Bel-abbes]']
    return base

# Table of point predictions
temps = [25, 30, 35, 40]
print('Predicted humidity (%)')
print(f'{"Temp":>6}  {"Bejaia":>8}  {"Sidi Bel-abbes":>15}')
for t in temps:
    print(f'{t:6.0f}  {predict_humid(t, "Bejaia"):8.1f}  {predict_humid(t, "Sidi Bel-abbes"):15.1f}')

# Monte-Carlo envelope (add residual noise)
np.random.seed(42)
sigma = np.sqrt(modelH.scale)  # residual standard error
n_sim = 500
t_grid = np.linspace(24, 40, 17)
sims_bejaia = np.array([predict_humid(t, 'Bejaia') + np.random.normal(0, sigma, n_sim)
                        for t in t_grid])

plt.figure(figsize=(7, 4.5))
plt.fill_between(t_grid, np.percentile(sims_bejaia, 5, axis=1),
                 np.percentile(sims_bejaia, 95, axis=1), alpha=0.25, label='90% MC band')
plt.plot(t_grid, [predict_humid(t, 'Bejaia') for t in t_grid], 'b-', lw=2, label='Point prediction')
plt.xlabel('Temperature (°C)')
plt.ylabel('Predicted Humidity (%)')
plt.title('Bejaia – Humidity prediction with Monte-Carlo uncertainty')
plt.legend()
plt.show()
plt.clf()

# What-if: raise humidity 10 points on the polynomial FFMC surface
def poly_ffmc(h):
    return modelP.params.iloc[0] + modelP.params.iloc[1]*h + modelP.params.iloc[2]*h**2

print('\nWhat-if on polynomial FFMC model:')
for h0 in [30, 50, 70]:
    print(f'  Humidity {h0}% → FFMC {poly_ffmc(h0):.1f}  |  +10% → FFMC {poly_ffmc(h0+10):.1f}  (Δ = {poly_ffmc(h0+10)-poly_ffmc(h0):.1f})')


## Optimisation & Production Tips

- **Single figure grids** instead of many `plt.show()/clf()` reduce notebook noise and make export to reports easier.
- Cache expensive objects (`corr_grid`, fitted models) and reuse them.
- Prefer formula API for readability; fall back to design matrices only when you need custom contrasts or sklearn pipelines.
- For repeated predictions, use `model.get_prediction(exog).summary_frame()` to obtain confidence/prediction intervals in one call.
- When the dataset grows (streaming weather stations), move the same formula into a `statsmodels` or `sklearn` pipeline and schedule nightly re-fits.


## Transferability – Inference Specialist playbook for any domain

1. **Load & inspect** – shape, missingness, value counts of key categoricals.
2. **Multicollinearity screen** – heatmap / VIF; drop or combine redundant predictors.
3. **EDA scatters** coloured by the categorical of interest.
4. **Baseline MLR** with main effects; write the equation and interpret each coefficient *in context*.
5. **Test interactions** when slopes look different; keep only those that change the scientific story.
6. **Polynomial / spline** terms when residual plots show curvature.
7. **Diagnostics** – residual vs fitted, QQ, leverage; report R² and residual SE.
8. **Simulation / sensitivity** – vary the continuous drivers and show stakeholders the resulting range of outcomes.
9. **Communicate** with a one-page summary that leads with the decision-relevant coefficient, not the p-value.

Swap `forests.csv` for a clinical, marketing, agricultural or financial CSV and the identical code skeleton produces a new, defensible analysis.


## Key Takeaways for the Inference Specialist

- Temperature has a strong negative association with humidity (≈ –2.4 pp per °C) that is parallel across the two Algerian regions; Sidi Bel-abbes is systematically drier.
- The relationship between temperature and FFMC **changes slope** once a fire has occurred – an interaction that a parallel-lines model would miss.
- FFMC vs humidity is clearly quadratic; a linear model would under-state the protective effect of high humidity.
- FWI is almost perfectly recoverable from ISI + BUI (R² ≈ 0.97), confirming the known physical definition.
- Always check collinearity among derived indices before putting them in the same regression.
- The whole workflow is domain-agnostic: change the data, keep the inference discipline.
